In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================
# MAE-baseline (ResNet-18) on PCam
# Masked reconstruction pretrain -> linear probe
# Protocol: 1 split x 3 seeds (n=3), label_fracs 1/5/10
# Outputs:
#   results_raw_resnet_mae_baseline.csv
#   results_summary_resnet_mae_baseline.csv + .tex
# ============================================================

import os, time, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from PIL import Image
import torchvision
from torchvision import transforms

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

OUT_DIR = "/kaggle/working/results"
os.makedirs(OUT_DIR, exist_ok=True)

SPLIT_ID = 0
SEEDS = [0,1,2]
LABEL_FRACS = [0.01, 0.05, 0.10]

# MAE-like baseline settings
MAE_EPOCHS = 10
MAE_BATCH = 128
MAE_LR = 1e-3
MASK_RATIO = 0.6

# Probe
PROBE_EPOCHS = 10
PROBE_BATCH = 256
PROBE_LR = 1e-3

PCAM_SAMPLE_FRAC = 0.2  # speed knob

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def append_row_csv(path, row: dict):
    df = pd.DataFrame([row])
    if os.path.exists(path):
        df.to_csv(path, mode="a", header=False, index=False)
    else:
        df.to_csv(path, index=False)

def summarize_mean_std(raw_csv_path, group_cols, metric_cols, out_csv_path, out_tex_path=None):
    df = pd.read_csv(raw_csv_path)
    g = df.groupby(group_cols)[metric_cols].agg(["mean","std","count"]).reset_index()
    g.columns = ["_".join([c for c in col if c]) for col in g.columns.values]
    g.to_csv(out_csv_path, index=False)
    if out_tex_path:
        g.to_latex(out_tex_path, index=False, float_format="%.4f")
    return g

# -----------------------
# Find PCam
# -----------------------
def find_pcam_root():
    for d in os.listdir("/kaggle/input"):
        root = os.path.join("/kaggle/input", d)
        if os.path.exists(os.path.join(root, "train_labels.csv")) and os.path.isdir(os.path.join(root, "train")):
            return root
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train_labels.csv" in files and "train" in dirs:
            return root
    raise FileNotFoundError("PCam not found. Add histopathologic-cancer-detection.")

PCAM_ROOT = find_pcam_root()
df = pd.read_csv(os.path.join(PCAM_ROOT, "train_labels.csv"))
train_dir = os.path.join(PCAM_ROOT, "train")
df["path"] = df["id"].apply(lambda x: os.path.join(train_dir, f"{x}.tif"))
if not os.path.exists(df["path"].iloc[0]):
    df["path"] = df["id"].apply(lambda x: os.path.join(train_dir, f"{x}.png"))
df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)

if 0 < PCAM_SAMPLE_FRAC < 1.0:
    df = df.sample(frac=PCAM_SAMPLE_FRAC, random_state=42).reset_index(drop=True)

paths = df["path"].values
y_all = df["label"].values.astype(int)
print("PCam:", len(paths))

# Split fixed
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
splits = list(skf.split(np.zeros(len(y_all)), y_all))
trainval_idx, test_idx = splits[SPLIT_ID]
y_trainval = y_all[trainval_idx]

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=123)
tr_rel, va_rel = next(sss.split(np.zeros(len(trainval_idx)), y_trainval))
train_idx = trainval_idx[tr_rel]
val_idx = trainval_idx[va_rel]

# -----------------------
# Datasets
# -----------------------
tf_img = transforms.Compose([
    transforms.Resize((96,96)),
    transforms.ToTensor(),
])

class ImgDataset(Dataset):
    def __init__(self, paths, labels=None):
        self.paths = paths
        self.labels = labels
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        x = tf_img(img)
        if self.labels is None:
            return x
        return x, int(self.labels[i])

# -----------------------
# MAE-like model: ResNet encoder + small decoder
# -----------------------
class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        m = torchvision.models.resnet18(weights=None)
        self.body = nn.Sequential(*list(m.children())[:-2])  # feature map [B,512,H/32,W/32]
    def forward(self, x):
        return self.body(x)

class ConvDecoder(nn.Module):
    def __init__(self, in_ch=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),
            nn.Sigmoid(),
        )
    def forward(self, z):
        return self.net(z)

def apply_random_mask(x, mask_ratio, seed):
    # x: [B,3,H,W] in [0,1]
    rng = torch.Generator(device=x.device)
    rng.manual_seed(seed)
    B, C, H, W = x.shape
    mask = torch.rand((B,1,H,W), generator=rng, device=x.device) < mask_ratio
    x_masked = x.clone()
    x_masked = x_masked.masked_fill(mask, 0.0)
    return x_masked, mask

def pretrain_mae(seed):
    set_all_seeds(seed)
    enc = ResNet18Encoder().to(device)
    dec = ConvDecoder(in_ch=512).to(device)

    ds = ImgDataset(paths[train_idx], labels=None)
    dl = DataLoader(ds, batch_size=MAE_BATCH, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

    opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=MAE_LR)

    enc.train(); dec.train()
    for ep in range(MAE_EPOCHS):
        total = 0.0; n=0
        for x in dl:
            x = x.to(device, non_blocking=True)
            x_masked, mask = apply_random_mask(x, MASK_RATIO, seed + ep)
            z = enc(x_masked)
            x_rec = dec(z)
            # loss only on masked pixels (focus reconstruction where removed)
            loss = ((x_rec - x) ** 2)
            loss = loss * mask
            loss = loss.mean()

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            total += float(loss.item()) * x.size(0); n += x.size(0)
        print(f"[MAE] seed{seed} ep{ep+1}/{MAE_EPOCHS} loss={total/max(1,n):.6f}")

    return enc

# -----------------------
# Linear probe on encoder features (global avg pooled)
# -----------------------
class Probe(nn.Module):
    def __init__(self, in_dim=512):
        super().__init__()
        self.fc = nn.Linear(in_dim, 1)
    def forward(self, h):
        return self.fc(h).squeeze(1)

@torch.no_grad()
def extract_feats(enc, loader):
    enc.eval()
    X=[]; Y=[]
    for x,y in loader:
        x = x.to(device, non_blocking=True)
        z = enc(x)                      # [B,512,h,w]
        h = z.mean(dim=[2,3])           # GAP -> [B,512]
        X.append(h.cpu().numpy())
        Y.append(y.numpy())
    return np.concatenate(X,0), np.concatenate(Y,0)

def train_probe(enc, label_frac, seed):
    set_all_seeds(seed + int(label_frac*1000))
    y_train = y_all[train_idx]
    idx = np.arange(len(train_idx))

    # stratified subsample
    sss = StratifiedShuffleSplit(n_splits=1, train_size=max(2, int(len(idx)*label_frac)), random_state=seed+7)
    sub_rel, _ = next(sss.split(np.zeros(len(idx)), y_train))
    sub_abs = train_idx[sub_rel]

    dl_tr = DataLoader(ImgDataset(paths[sub_abs], y_all[sub_abs]), batch_size=PROBE_BATCH, shuffle=True, num_workers=2, pin_memory=True)
    dl_va = DataLoader(ImgDataset(paths[val_idx], y_all[val_idx]), batch_size=PROBE_BATCH, shuffle=False, num_workers=2, pin_memory=True)
    dl_te = DataLoader(ImgDataset(paths[test_idx], y_all[test_idx]), batch_size=PROBE_BATCH, shuffle=False, num_workers=2, pin_memory=True)

    Xtr, ytr = extract_feats(enc, dl_tr)
    Xva, yva = extract_feats(enc, dl_va)
    Xte, yte = extract_feats(enc, dl_te)

    Xtr_t = torch.tensor(Xtr, dtype=torch.float32, device=device)
    ytr_t = torch.tensor(ytr, dtype=torch.float32, device=device)
    Xva_t = torch.tensor(Xva, dtype=torch.float32, device=device)
    yva_t = torch.tensor(yva, dtype=torch.float32, device=device)

    probe = Probe(in_dim=Xtr.shape[1]).to(device)
    opt = torch.optim.Adam(probe.parameters(), lr=PROBE_LR)
    lossfn = nn.BCEWithLogitsLoss()

    best = 1e9
    best_state = None
    for ep in range(PROBE_EPOCHS):
        probe.train()
        perm = torch.randperm(Xtr_t.size(0), device=device)
        for i in range(0, Xtr_t.size(0), PROBE_BATCH):
            ii = perm[i:i+PROBE_BATCH]
            logits = probe(Xtr_t[ii])
            loss = lossfn(logits, ytr_t[ii])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

        probe.eval()
        with torch.no_grad():
            vloss = float(lossfn(probe(Xva_t), yva_t).item())
        if vloss < best:
            best = vloss
            best_state = {k:v.detach().cpu() for k,v in probe.state_dict().items()}
        print(f"[Probe] ep{ep+1}/{PROBE_EPOCHS} val_loss={vloss:.4f}")

    probe.load_state_dict(best_state, strict=True)

    # test metrics
    probe.eval()
    Xte_t = torch.tensor(Xte, dtype=torch.float32, device=device)
    with torch.no_grad():
        logits = probe(Xte_t).cpu().numpy()
        probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    acc = accuracy_score(yte, preds)
    f1  = f1_score(yte, preds)
    try:
        auc = roc_auc_score(yte, probs)
    except Exception:
        auc = float("nan")

    # simple calibration
    brier = float(np.mean((probs - yte.astype(float))**2))
    # ECE (quick)
    def ece(p, y, n_bins=15):
        p = np.clip(p, 1e-7, 1-1e-7)
        bins = np.linspace(0,1,n_bins+1)
        e=0.0
        for i in range(n_bins):
            lo,hi=bins[i],bins[i+1]
            m=(p>=lo)&(p<hi)
            if m.sum()==0: continue
            e += (m.sum()/len(y))*abs(y[m].mean() - p[m].mean())
        return float(e)
    ece_v = ece(probs, yte)

    return {"acc": float(acc), "auc": float(auc), "f1": float(f1), "ece": float(ece_v), "brier": float(brier)}

# -----------------------
# RUN
# -----------------------
RAW_PATH = f"{OUT_DIR}/results_raw_resnet_mae_baseline.csv"

for seed in SEEDS:
    enc = pretrain_mae(seed)
    for frac in LABEL_FRACS:
        metrics = train_probe(enc, frac, seed)
        row = {
            "exp":"mae_baseline",
            "backbone":"resnet18",
            "split_id":SPLIT_ID,
            "seed":seed,
            "label_frac":frac,
            **metrics
        }
        append_row_csv(RAW_PATH, row)
        print("seed", seed, "frac", frac, "metrics", metrics)

summarize_mean_std(
    RAW_PATH,
    group_cols=["exp","backbone","label_frac"],
    metric_cols=["acc","auc","f1","ece","brier"],
    out_csv_path=f"{OUT_DIR}/results_summary_resnet_mae_baseline.csv",
    out_tex_path=f"{OUT_DIR}/results_summary_resnet_mae_baseline.tex"
)

print("Saved:", RAW_PATH)
